# Limpieza de datos

Se realiza la limpieza de datos y se genera la tabla de ventas en schema de silver

In [0]:
T_BRONZE = 'blur_proyecto_final.bronze.sales_raw'
T_SILVER = 'blur_proyecto_final.silver.sales_silver'
T_VENTAS_CUANRENTENA = 'blur_proyecto_final.silver.sales_silver_cuarentena'

spark.sql(f"DROP TABLE IF EXISTS {T_SILVER}")
spark.sql(f"DROP TABLE IF EXISTS {T_VENTAS_CUANRENTENA}")

In [0]:
T_BRONZE = 'blur_proyecto_final.bronze.sales_raw'
df_sales = spark.read.table(T_BRONZE)
filas_bronze = df_sales.count()
print(filas_bronze)

In [0]:
display(spark.table(f"{T_BRONZE}").limit(5))

## Se consulta el ESQUEMA
Se verifica el esquema para revisar los campos y tipos de datos

In [0]:
df_sales.printSchema()

## # Contar valores NULL
Contar cuantos valores null hay por cada columna

In [0]:
from pyspark.sql import functions as F

df_sales.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_sales.columns
]).show()

In [0]:
from pyspark.sql import functions as F
df_silver_ventas = ( df_sales                    
                     .select(
                        F.col("factura").alias("factura"),
                        F.col("cliente").alias("cliente"),
                        F.col("cantidad").alias("cantidad"),
                        F.col("fecha").alias("fecha"),
                        F.col("monto").alias("monto")
                        )
                      .withColumn("factura",F.regexp_replace(F.col("factura"), "^C", "") ) 
                      .withColumn("monto", F.regexp_replace(F.col("monto"), "[,]", "."))
                      .withColumn("monto", F.col("monto").cast("double") )                
                      .withColumn("fecha", F.to_timestamp(F.col("fecha"), "M/d/yyyy H:mm:ss"))
                      .withColumn("date", F.to_date(F.col("fecha")) )
                      .withColumn("cliente", F.coalesce(F.col("cliente"), F.lit("sin_cliente")))
                      .withColumn("cantidad",F.col("cantidad").cast("int") )
                      .filter((F.col("cantidad") >= 0) & (F.col("monto") >= 0))
                      .withColumn("fecha_ingesta", F.current_timestamp())
                      .withColumn("usuario_ingesta", F.current_user())
                      .withColumn("origen",F.lit("kaggle"))
                      .dropDuplicates(["factura"])
)

df_ventas_cuarentena = ( df_sales                    
                     .select(
                        F.col("factura").alias("factura"),
                        F.col("cliente").alias("cliente"),
                        F.col("cantidad").alias("cantidad"),
                        F.col("fecha").alias("fecha"),
                        F.col("monto").alias("monto")
                        )
                      .withColumn("monto", F.regexp_replace(F.col("monto"), "[,]", "."))
                      .withColumn("monto", F.col("monto").cast("double") )                    
                      .withColumn("cantidad",F.col("cantidad").cast("int") )   
                      .filter((F.col("cantidad") < 0) | (F.col("monto") < 0))  
                      .withColumn("fecha_ingesta", F.current_timestamp())
                      .withColumn("usuario_ingesta", F.current_user())
                      .withColumn("origen",F.lit("kaggle"))

)

print(df_silver_ventas.count())
print(df_ventas_cuarentena.count())
print( df_silver_ventas.count() +  df_ventas_cuarentena.count())






## Verificar cantidad de registros
Se verifica que la cantidad de registros en el dataframe de silver mas el dataframe de cuarentena corresponda con la cantidad de registros en la tabla de bronce.

In [0]:

print(f"Filas en silver: {df_silver_ventas.count()}")
print(f"Filas en silver cuarentena: {df_ventas_cuarentena.count()}")
print(f"Suma filas en silver mas cuarentena: {df_ventas_cuarentena.count() + df_silver_ventas.count()}")
filas_bronze = spark.table(f"{T_BRONZE}").count()
print(f"Filas en {T_BRONZE}: {filas_bronze}")

In [0]:
filas_silver = spark.table(f"{T_SILVER}").count()
print(f"Filas en {T_SILVER}: {filas_silver}")

## Creacion tabla en Silver
Despues de la limpieza de los datos se crea la tabla en silver

In [0]:
df_vacio = df_silver_ventas.where("1 = 0")
df_vacio.write.mode("overwrite").saveAsTable("blur_proyecto_final.silver.sales_silver")


In [0]:
df_silver_ventas.write.mode("overwrite").saveAsTable("blur_proyecto_final.silver.sales_silver_t")

In [0]:
%sql
MERGE INTO blur_proyecto_final.silver.sales_silver AS sales
USING blur_proyecto_final.silver.sales_silver_t AS new
ON sales.factura = new.factura
WHEN MATCHED THEN
UPDATE SET 
sales.cliente = new.cliente,
sales.cantidad = new.cantidad,
sales.fecha = new.fecha,
sales.monto = new.monto,
sales.date = new.date,
sales.fecha_ingesta = new.fecha_ingesta,
sales.usuario_ingesta = new.usuario_ingesta,
sales.origen = new.origen       
WHEN NOT MATCHED THEN
INSERT (factura, cliente, cantidad, fecha, monto, date, fecha_ingesta, usuario_ingesta, origen)
VALUES (new.factura, new.cliente, new.cantidad, new.fecha, new.monto, new.date, new.fecha_ingesta, new.usuario_ingesta, new.origen)